# Gemini 3.1 Flash Live Preview Cookbook

**Model:** `gemini-3.1-flash-live-preview`  
**Latest update:** March 2026  
**Provider:** Google DeepMind / Gemini API

Gemini 3.1 Flash Live Preview is a low-latency, audio-to-audio model optimized for real-time dialogue and voice-first AI applications with acoustic nuance detection, numeric precision, and multimodal awareness.

**Key specs from docs:**
- Input modalities: text, audio, images, video
- Output modalities: text, audio
- Input token limit: 131,072
- Output token limit: 65,536
- Knowledge cutoff: January 2025
- Protocol: WebSocket (stateful, bidirectional)
- Audio specs: Input 16-bit PCM 16kHz, Output 16-bit PCM 24kHz

**Best-fit workloads:**
- Real-time voice assistants and conversational AI
- Customer support with voice interaction
- Accessibility applications
- Interactive gaming and NPCs
- Smart home and IoT voice interfaces
- Multimodal dialogue with audio, text, and vision

**Table of Contents:**
1. Setup and Installation
2. Basic Text Conversation
3. Audio Streaming and Real-time Dialogue
4. Video Input with Live Conversation
5. Tool Use and Function Calling
6. Session Management and State
7. Real-world Examples
8. Best Practices and Optimization

## 1. Setup and Installation

In [76]:
# Install required packages
# Note: pyaudio is optional - it requires C++ build tools on Windows
# The Live API handles audio streaming, so pyaudio is not essential for this cookbook
import subprocess
import sys

install_cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--disable-pip-version-check",
    "google-genai",
    "numpy",
]

result = subprocess.run(install_cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Core packages installed successfully")
else:
    print("❌ Package installation failed")
    if result.stderr:
        print(result.stderr.strip())

# Optional packages (install if needed for your use case):
# For audio file handling (optional): pip install soundfile
# For microphone input on Windows: pip install pyaudio-wheels
# For microphone input on Mac/Linux: pip install pyaudio

✓ Core packages installed successfully


In [40]:
import asyncio
import os
from typing import Union
from google import genai
from google.genai import types

# Configuration
API_KEY = os.environ.get('GEMINI_API_KEY')
if not API_KEY:
    print("⚠️  Warning: GEMINI_API_KEY environment variable not set")
    print("Set it with: os.environ['GEMINI_API_KEY'] = 'your-api-key'")
    API_KEY = "YOUR_API_KEY_HERE"

MODEL = 'gemini-3.1-flash-live-preview'

# Initialize client only if API key is available
if API_KEY != "YOUR_API_KEY_HERE":
    client = genai.Client(api_key=API_KEY)
    print(f"✓ Using model: {MODEL}")
    print(f"✓ API Key configured: {bool(API_KEY)}")
else:
    client = None
    print(f"✓ Model: {MODEL}")
    print(f"⚠️  API Key not configured")

✓ Using model: gemini-3.1-flash-live-preview
✓ API Key configured: True


## 2. Basic Text Conversation

Start with a simple real-time text conversation using WebSockets.

### Optional Dependencies for Audio Capture (Local Microphone)

If you want to capture audio from your microphone and send it to the Live API:

**Windows:**
```bash
pip install pyaudio-wheels
```

**macOS:**
```bash
brew install portaudio
pip install pyaudio
```

**Linux:**
```bash
sudo apt-get install portaudio19-dev
pip install pyaudio
```

**Note:** The cookbook examples work perfectly without these - the Live API handles all audio streaming. These are only needed if you want to capture audio from a local device.


In [59]:
async def simple_text_conversation():
    """
    Demonstrates a basic text-based conversation with the Live API.
    
    This is useful for:
    - Testing the connection
    - Text-only chat interfaces
    - Server-side processing
    
    Usage: await simple_text_conversation()
    """
    if not client:
        print("❌ API key not configured. Skipping example.")
        return
    
    config = {
        "response_modalities": ["TEXT"],
        "system_instruction": "You are a helpful assistant. Keep responses concise."
    }
    
    try:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            print("✓ Session started. Sending text...")
            
            # Send a text message
            await session.send_realtime_input(text="What are the key features of the Live API?")
            
            print("\n📝 Response:")
            # Receive and display responses
            async for response in session.receive():
                if response.server_content:
                    # Check for model turn content
                    if response.server_content.model_turn:
                        for part in response.server_content.model_turn.parts:
                            if hasattr(part, 'text') and part.text:
                                print(f"{part.text}", end="", flush=True)
            print("\n")
    except Exception as e:
        print(f"❌ Error: {e}")

print("✓ simple_text_conversation() function defined")

✓ simple_text_conversation() function defined


## 3. Audio Streaming and Real-time Dialogue

The Live API excels at real-time audio streaming with low latency. Here's how to send and receive audio.

In [60]:
def get_audio_config(voice: str = "Ember", response_modality: Union[str, list] = "AUDIO") -> dict:
    """
    Helper to configure audio settings for the Live API.
    
    Args:
        voice: Voice profile ("Ember", "Breeze", "Cove", "Juniper", "Orbit", "Sage")
        response_modality: "AUDIO", "TEXT", or ["AUDIO", "TEXT"] for both
    
    Returns:
        Configuration dict for Live API session
    """
    modalities = [response_modality] if isinstance(response_modality, str) else response_modality
    
    config = {
        "response_modalities": modalities,
        "system_instruction": "You are a friendly and concise conversational AI."
    }
    
    # Add voice config only if using audio
    if "AUDIO" in modalities:
        config["speech_config"] = {
            "voice_config": {
                "prebuilt_voice_config": {
                    "voice_name": voice
                }
            }
        }
    
    return config

# Test the config helper
test_config = get_audio_config(voice="Ember", response_modality=["AUDIO", "TEXT"])
print("✓ Audio config helper ready")
print(f"  Sample config keys: {list(test_config.keys())}")

✓ Audio config helper ready
  Sample config keys: ['response_modalities', 'system_instruction', 'speech_config']


In [61]:
async def audio_streaming_example():
    """
    Demonstrates audio streaming with the Live API.
    
    Note: This example shows the pattern. For actual audio input,
    you would need to capture audio from a microphone or file.
    Audio must be: raw 16-bit PCM, 16kHz sample rate, little-endian
    
    Usage: await audio_streaming_example()
    """
    if not client:
        print("❌ API key not configured. Skipping example.")
        return
    
    config = get_audio_config(voice="Ember", response_modality=["AUDIO", "TEXT"])
    
    try:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            print("✓ Audio session started")
            
            # Example: Send text to demonstrate the pattern
            # In production, you'd stream actual audio chunks here
            user_input = "Tell me a short joke about technology."
            print(f"\n📤 Sending: {user_input}")
            await session.send_realtime_input(text=user_input)
            
            print("📥 Receiving response...")
            async for response in session.receive():
                if response.server_content:
                    # Audio output chunks
                    if response.server_content.model_turn:
                        for part in response.server_content.model_turn.parts:
                            if part.inline_data:
                                audio_size = len(part.inline_data.data) if part.inline_data.data else 0
                                if audio_size > 0:
                                    print(f"[Audio chunk: {audio_size} bytes]")
                            elif hasattr(part, 'text') and part.text:
                                print(f"📝 {part.text}", end="", flush=True)
            print("\n")
    except Exception as e:
        print(f"❌ Error: {e}")

print("✓ audio_streaming_example() function defined")

✓ audio_streaming_example() function defined


### Audio Input Helper Functions

Utilities for handling audio input/output with the Live API.

In [62]:
def create_audio_blob(audio_bytes: bytes, sample_rate: int = 16000) -> types.Blob:
    """
    Create a Blob object from raw audio bytes.
    
    Args:
        audio_bytes: Raw PCM audio data (16-bit, little-endian)
        sample_rate: Sample rate in Hz (typically 16000)
    
    Returns:
        types.Blob formatted for Live API
    """
    mime_type = f"audio/pcm;rate={sample_rate}"
    return types.Blob(data=audio_bytes, mime_type=mime_type)


async def process_audio_chunks(
    session,
    audio_chunks: list[bytes],
    chunk_delay_ms: int = 100
) -> None:
    """
    Send audio chunks to the Live API in a streaming fashion.
    
    Args:
        session: Active Live API session
        audio_chunks: List of raw PCM audio chunks
        chunk_delay_ms: Delay between chunks in milliseconds
    """
    for chunk in audio_chunks:
        blob = create_audio_blob(chunk)
        await session.send_realtime_input(audio=blob)
        await asyncio.sleep(chunk_delay_ms / 1000)


def extract_audio_output(response_content) -> bytes:
    """
    Extract audio data from a Live API response.
    
    Args:
        response_content: ServerContent from Live API response
    
    Returns:
        Audio bytes (24kHz PCM) or empty bytes if no audio
    """
    audio_data = b''
    if response_content and response_content.model_turn:
        for part in response_content.model_turn.parts:
            if hasattr(part, 'inline_data') and part.inline_data:
                audio_data += part.inline_data.data or b''
    return audio_data


print("Audio helper functions defined.")

Audio helper functions defined.


## 4. Video Input with Live Conversation

The Live API supports video input alongside audio/text for multimodal interactions.

In [63]:
def create_video_blob(image_bytes: bytes, mime_type: str = "image/jpeg") -> types.Blob:
    """
    Create a Blob object from image/video frame data.
    
    Args:
        image_bytes: JPEG or PNG encoded image data
        mime_type: "image/jpeg" or "image/png"
    
    Returns:
        types.Blob formatted for Live API
    """
    return types.Blob(data=image_bytes, mime_type=mime_type)


async def multimodal_video_conversation():
    """
    Demonstrates video input with audio conversation.
    
    Max frame rate: 1 frame per second
    Supported formats: JPEG, PNG
    
    Usage: await multimodal_video_conversation()
    """
    if not client:
        print("❌ API key not configured. Skipping example.")
        return
    
    config = get_audio_config(voice="Cove", response_modality=["AUDIO", "TEXT"])
    
    try:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            print("✓ Video+Audio session started")
            
            # Example: Send a text query about a video frame
            user_query = "If you were looking at an image, what would you tell me about it?"
            print(f"Query: {user_query}")
            
            # In production, you would:
            # 1. Capture video frame as JPEG: frame_bytes = capture_frame()
            # 2. Send it: await session.send_realtime_input(video=create_video_blob(frame_bytes))
            # 3. Send audio/text at ~1 frame per second max
            
            await session.send_realtime_input(text=user_query)
            
            async for response in session.receive():
                if response.server_content and response.server_content.model_turn:
                    for part in response.server_content.model_turn.parts:
                        if hasattr(part, 'text') and part.text:
                            print(f"{part.text}", end="", flush=True)
            print("\n")
    except Exception as e:
        print(f"❌ Error: {e}")

print("✓ multimodal_video_conversation() function defined")

✓ multimodal_video_conversation() function defined


## 5. Tool Use and Function Calling

The Live API supports synchronous function calling for dynamic interactions.

In [64]:
# Define tool implementation functions
def get_weather(location: str) -> dict:
    """
    Get current weather for a location.
    
    Args:
        location: City name or coordinates
    
    Returns:
        Weather information dict
    """
    return {
        "location": location,
        "temperature": 72,
        "condition": "Partly Cloudy",
        "humidity": 65
    }


def calculate(expression: str) -> Union[float, dict]:
    """
    Perform mathematical calculations.
    
    Args:
        expression: Math expression (e.g., '2+2*3')
    
    Returns:
        Calculation result or error dict
    """
    try:
        result = eval(expression)
        return result if isinstance(result, (int, float)) else float(result)
    except Exception as e:
        return {"error": f"Calculation failed: {str(e)}", "result": 0.0}


# Tool functions dictionary for easy lookup
tool_functions = {
    "get_weather": get_weather,
    "calculate": calculate
}

# Google Genai SDK tools - defined as callables
# Note: To use in Live API, tools must be configured with proper OpenAPI schemas
# See conversation_with_tools() for the pattern
TOOLS = [get_weather, calculate]

print("✓ Tool functions defined:", [t.__name__ for t in TOOLS])
print("  (Tools require OpenAPI schema configuration for Live API use)")

✓ Tool functions defined: ['get_weather', 'calculate']
  (Tools require OpenAPI schema configuration for Live API use)


In [65]:
async def conversation_with_tools():
    """
    Demonstrates a conversation with tool functions available.
    
    Note: The Live API requires tools to be configured with proper OpenAPI schemas.
    Tool functions defined in this cookbook:
    - get_weather(location: str) -> dict
    - calculate(expression: str) -> Union[float, dict]
    
    For production use with tools:
    1. Define tools with complete OpenAPI schema (name, description, parameters)
    2. Pass tools list to config["tools"] 
    3. Handle tool_call responses from the model
    4. Execute tools and send results back via send_tool_response()
    
    Usage: await conversation_with_tools()
    """
    if not client:
        print("❌ API key not configured. Skipping example.")
        return
    
    config = get_audio_config(voice="Ember", response_modality=["TEXT"])
    # Note: To use tools, they must be configured with OpenAPI schema
    # config["tools"] = [...] # Would go here with proper schema
    
    try:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            print("✓ Session started (tools available for extension)")
            
            # Send a query that could use tools
            user_input = "Can you tell me the weather in San Francisco and calculate 2+2?"
            print(f"\n👤 User: {user_input}")
            await session.send_realtime_input(text=user_input)
            
            response_count = 0
            async for response in session.receive():
                # Check for tool calls (if tools were configured)
                if hasattr(response, 'tool_call') and response.tool_call:
                    print(f"\n[🔧 Model requesting tool calls]")
                    function_responses = []
                    
                    for fc in response.tool_call.function_calls:
                        func_name = fc.name
                        func_args = fc.args if hasattr(fc, 'args') else {}
                        func_id = fc.id
                        
                        print(f"  Calling: {func_name}({func_args})")
                        
                        # Execute the function
                        if func_name in tool_functions:
                            result = tool_functions[func_name](**func_args)
                        else:
                            result = {"error": f"Unknown function: {func_name}"}
                        
                        print(f"  Result: {result}")
                        
                        # Prepare response
                        function_responses.append(types.FunctionResponse(
                            name=func_name,
                            id=func_id,
                            response=result
                        ))
                    
                    # Send tool responses back to the session
                    print(f"  Sending {len(function_responses)} tool response(s)...")
                    await session.send_tool_response(function_responses=function_responses)
                
                # Process text response
                if response.server_content and response.server_content.model_turn:
                    for part in response.server_content.model_turn.parts:
                        if hasattr(part, 'text') and part.text:
                            if response_count == 0:
                                print(f"\n🤖 Assistant: ", end="")
                                response_count += 1
                            print(part.text, end="", flush=True)
            print("\n")
    except Exception as e:
        print(f"❌ Error: {e}")

print("✓ conversation_with_tools() function defined")

✓ conversation_with_tools() function defined


## 6. Session Management and State

Managing long-running conversations and maintaining context.

In [66]:
class LiveConversationManager:
    """
    Manages a conversation session with the Live API.
    Handles connection lifecycle, message history, and state.
    """
    
    def __init__(
        self,
        api_key: str,
        model: str = "gemini-3.1-flash-live-preview",
        voice: str = "Ember",
        system_instruction: str = None
    ):
        self.client = genai.Client(api_key=api_key)
        self.model = model
        self.voice = voice
        self.system_instruction = system_instruction or "You are a helpful assistant."
        self.session = None
        self.message_history = []
    
    async def connect(self):
        """Establish a connection to the Live API."""
        config = {
            "response_modalities": ["TEXT"],
            "system_instruction": self.system_instruction,
        }
        
        if "AUDIO" in config.get("response_modalities", []):
            config["speech_config"] = {
                "voice_config": {
                    "prebuilt_voice_config": {
                        "voice_name": self.voice
                    }
                }
            }
        
        self.session = await self.client.aio.live.connect(
            model=self.model,
            config=config
        )
        self.session = await self.session.__aenter__()
        return self.session
    
    async def send_message(self, text: str) -> str:
        """Send a message and receive a response."""
        if not self.session:
            raise RuntimeError("Not connected. Call connect() first.")
        
        self.message_history.append({"role": "user", "content": text})
        await self.session.send_realtime_input(text=text)
        
        full_response = ""
        async for response in self.session.receive():
            if response.server_content and response.server_content.model_turn:
                for part in response.server_content.model_turn.parts:
                    if hasattr(part, 'text') and part.text:
                        full_response += part.text
        
        self.message_history.append({"role": "assistant", "content": full_response})
        return full_response
    
    async def disconnect(self):
        """Close the session."""
        if self.session:
            try:
                await self.session.__aexit__(None, None, None)
            except:
                pass
            self.session = None
    
    def get_history(self) -> list:
        """Retrieve conversation history."""
        return self.message_history

print("✓ LiveConversationManager class defined")

✓ LiveConversationManager class defined


## 7. Real-world Examples

Practical examples for common use cases.

### Example 1: Simple Voice Assistant

In [67]:
async def voice_assistant_loop():
    """
    A simple interactive voice assistant that processes
    multiple turns of conversation.
    
    Usage: await voice_assistant_loop()
    """
    if not client:
        print("❌ API key not configured. Skipping example.")
        return
    
    config = {
        "response_modalities": ["TEXT"],
        "system_instruction": "You are a helpful voice assistant. Be concise and friendly."
    }
    
    print("🎤 Voice Assistant Starting...")
    
    try:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            # Simulate a multi-turn conversation
            queries = [
                "Hello! What's your favorite programming language?",
                "What time is it?",
                "Tell me a fun fact about space."
            ]
            
            for i, query in enumerate(queries, 1):
                print(f"\n[Turn {i}]")
                print(f"👤 User: {query}")
                
                await session.send_realtime_input(text=query)
                
                response_text = ""
                async for response in session.receive():
                    if response.server_content:
                        if response.server_content.model_turn:
                            for part in response.server_content.model_turn.parts:
                                if hasattr(part, 'text') and part.text:
                                    response_text += part.text
                
                print(f"🤖 Assistant: {response_text}")
    except Exception as e:
        print(f"❌ Error: {e}")

print("✓ voice_assistant_loop() function defined")

✓ voice_assistant_loop() function defined


### Example 2: Customer Support Agent with Context

In [68]:
async def customer_support_agent():
    """
    A customer support agent that maintains context
    and demonstrates tool-enabled conversation pattern.
    
    This example shows the structure for a support agent.
    To enable tools, configure them with proper OpenAPI schemas.
    
    Usage: await customer_support_agent()
    """
    support_context = """
    You are a helpful customer support agent for TechCorp Inc.
    You have access to tools to look up order status and account information.
    Be empathetic, professional, and solve problems quickly.
    """
    
    config = {
        "response_modalities": ["TEXT"],
        "system_instruction": support_context,
        # "tools": [...]  # Would pass tools here with proper OpenAPI schema
    }
    
    print("📞 Customer Support Agent Starting...")
    
    try:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            # Simulate customer inquiry
            customer_inquiry = "I need help with my order status."
            print(f"👤 Customer: {customer_inquiry}")
            
            await session.send_realtime_input(text=customer_inquiry)
            
            async for response in session.receive():
                if hasattr(response, 'tool_call') and response.tool_call:
                    print("[Processing tool request...]")
                    # Tool handling logic would go here
                
                if response.server_content and response.server_content.model_turn:
                    for part in response.server_content.model_turn.parts:
                        if hasattr(part, 'text') and part.text:
                            print(f"🤖 Support Agent: {part.text}")
    except Exception as e:
        print(f"❌ Error: {e}")

print("✓ customer_support_agent() function defined")

✓ customer_support_agent() function defined


## Additional Resources

- **Live API Overview**: https://ai.google.dev/gemini-api/docs/live-api
- **GenAI SDK Guide**: https://ai.google.dev/gemini-api/docs/live-api/get-started-sdk
- **WebSocket API Reference**: https://ai.google.dev/api/live
- **Live API Capabilities**: https://ai.google.dev/gemini-api/docs/live-api/capabilities
- **Tool Use**: https://ai.google.dev/gemini-api/docs/live-api/tools
- **Session Management**: https://ai.google.dev/gemini-api/docs/live-api/session-management
- **Ephemeral Tokens**: https://ai.google.dev/gemini-api/docs/live-api/ephemeral-tokens
- **Best Practices**: https://ai.google.dev/gemini-api/docs/live-api/best-practices
- **GitHub Examples**: https://github.com/google-gemini/gemini-live-api-examples
- **AI Studio (Try Live)**: https://aistudio.google.com/live?model=gemini-3.1-flash-live-preview